# 01. EDA dữ liệu review
Khám phá vùng clean (output B2): khối lượng, ngôn ngữ, giờ chơi, review đáng ngờ, lệch dữ liệu.
Chạy sau `python pipeline.py batch`.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Kết quả của profile local (python pipeline.py batch) nằm ở data/lake; đổi bằng biến LAKE_DIR
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LAKE = Path(os.getenv("LAKE_DIR", ROOT / "data" / "lake"))
CLEAN, CURATED = LAKE / "clean", LAKE / "curated"
read = pd.read_parquet

In [ ]:
reviews = read(CLEAN / "reviews", columns=["recommendationid", "appid", "language", "voted_up", "playtime_at_review",
                                           "is_suspicious", "suspicious_reason", "source", "review_length", "dt"])
reviews["dt"] = pd.to_datetime(reviews.dt.astype(str))  # cột phân vùng đọc ra dạng category
print(f"{len(reviews):,} review, {reviews.appid.nunique()} game, từ {reviews.dt.min():%Y-%m-%d} tới {reviews.dt.max():%Y-%m-%d}")
reviews.groupby("source").size().rename("reviews")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
reviews.groupby(reviews.dt.dt.to_period("W")).size().plot(ax=ax[0], title="Review theo tuần")
reviews.language.value_counts().head(10).plot.barh(ax=ax[1], title="Top ngôn ngữ")
plt.tight_layout()

In [ ]:
hours = (reviews.playtime_at_review / 60).clip(upper=500)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
hours.plot.hist(bins=50, ax=ax[0], log=True, title="Giờ chơi lúc viết review (cắt ở 500h)")
reviews.groupby("voted_up").review_length.median().plot.bar(ax=ax[1], title="Độ dài review trung vị theo nhãn")
plt.tight_layout()

In [ ]:
# Review đáng ngờ (luật §6.3): giờ chơi thấp, tài khoản 1 review, nhận game miễn phí, nội dung trùng lặp
print(f"Tỷ lệ đáng ngờ: {reviews.is_suspicious.mean():.1%}")
reviews.loc[reviews.is_suspicious, "suspicious_reason"].str.split(",").explode().value_counts()

In [ ]:
# Lệch dữ liệu (skew): vài game chiếm phần lớn review
per_game = reviews.groupby("appid").size().sort_values(ascending=False)
print(f"Top 10% game chiếm {per_game.head(max(1, len(per_game) // 10)).sum() / per_game.sum():.1%} số review")
per_game.head(20).plot.bar(figsize=(13, 3), title="Top 20 game theo số review")